# Investigacion de posible tunelizacion DNS

## Objetivo

Analizar patrones compatibles con C2, tunelizacion o exfiltracion por DNS.

## Entradas esperadas

- IP, hostname o dominio raiz.
- Ventana de tiempo.

## Requisitos

- Acceso al area de trabajo de Microsoft Sentinel.
- Funciones KQL publicadas: `fn_Normalize_Windows_DHCP`, `fn_Normalize_Windows_DNS`, `fn_Correlate_DHCP_DNS`.
- Paquetes Python sugeridos: `msticpy`, `pandas`, `matplotlib`, `plotly`, `networkx` segun el notebook.

## Secciones

1. Longitud de consultas.
2. Proporcion TXT.
3. Subdominios distintos.
4. Ejemplos de consultas.


In [ ]:
# Configuracion general - ajustar antes de ejecutar
workspace_id = "REEMPLAZAR_CON_WORKSPACE_ID"
tenant_id = "REEMPLAZAR_CON_TENANT_ID"

# Conexion sugerida con MSTICPy
# import msticpy as mp
# mp.init_notebook(namespace=globals())
# qry_prov = mp.QueryProvider("MSSentinel")
# qry_prov.connect(workspace=workspace_id, tenant_id=tenant_id)


In [ ]:
query_tunnel = """
let Lookback = 7d;
let TargetIp = "REEMPLAZAR_CON_IP_OPCIONAL";
let TargetDomain = tolower("REEMPLAZAR_CON_DOMINIO_OPCIONAL");
fn_Correlate_DHCP_DNS(Lookback)
| where IsReverseLookup == false
| where TargetIp == "REEMPLAZAR_CON_IP_OPCIONAL" or ClientIp == TargetIp
| where TargetDomain == "reemplazar_con_dominio_opcional" or QueryRootDomain == TargetDomain
| summarize TotalQueries=count(), DistinctQueries=dcount(QueryName), AvgQueryLength=avg(QueryLength), MaxQueryLength=max(QueryLength), AvgLabelCount=avg(LabelCount), MaxLabelLength=max(MaxLabelLength), TxtQueries=countif(toupper(QueryType) == "TXT"), SampleQueries=make_set(QueryName, 20) by ClientIp, HostName, ClientMac, QueryRootDomain
| extend TxtRatio = todouble(TxtQueries) / todouble(TotalQueries)
| order by MaxQueryLength desc, DistinctQueries desc
"""
# tunnel_df = qry_prov.exec_query(query_tunnel)
print(query_tunnel)


In [ ]:
# Analisis local sugerido si tunnel_df esta cargado
# tunnel_df["RiskScore"] = (
#     (tunnel_df["MaxQueryLength"] >= 120).astype(int) * 2 +
#     (tunnel_df["MaxLabelLength"] >= 50).astype(int) * 2 +
#     (tunnel_df["TxtRatio"] >= 0.20).astype(int) * 2 +
#     (tunnel_df["DistinctQueries"] >= 80).astype(int)
# )
# tunnel_df.sort_values("RiskScore", ascending=False).head(20)


## Resumen para incidente

Documentar aqui:

- Hallazgos principales.
- Entidades relevantes: IP, hostname, direccion MAC, dominio.
- Evidencia KQL usada.
- Recomendacion: cerrar, monitorear, escalar o contener.
